<a href="https://colab.research.google.com/github/Lau-Tisca/FlyRank_ML_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lau-Tisca/FlyRank_ML_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

For **Lane 2: Refresh / Content Opportunity Scoring**, the primary formulation is a **Ranking and Scoring task**.

* **Why Ranking/Scoring instead of Binary Classification?** While we may train an underlying model using binary classification (e.g., predicting whether a page's traffic will decline or not), the final operational output must be a prioritized list. Human content editors and SEO specialists have finite weekly attention and action capacity. Surfacing a binary "yes/no decline" label is unhelpful because it does not tell the editor *which* page to click first. A continuous priority score (scaled 0 to 100) allows the system to order pages dynamically, placing the highest-urgency, highest-exposure candidate refreshes at the absolute top of the review queue.

In [ ]:
import os
import pandas as pd

# Path routing: local relative path first, then raw GitHub fallback
local_path = "../../data/raw/content_refresh_anonymized.csv"
fallback_url = "https://raw.githubusercontent.com/Lau-Tisca/FlyRank_ML_1/main/data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print("Successfully loaded data from local path.")
else:
    df = pd.read_csv(fallback_url)
    print("Successfully loaded data from raw GitHub URL.")

# Illustrate the massive range of scale in page impressions, proving why ranking is essential
print(f"Total Page Inventory: {len(df):,}")
print("\nImpression Volume Skew (90d):")
print(df['impressions_90d'].describe())

Successfully loaded data from raw GitHub URL.
Total Page Inventory: 30,000

Impression Volume Skew (90d):
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

* **What we predict in the starter playground (The Proxy Label):**
  We utilize a binary proxy label: `is_declining_label = (trend_direction == "down")`. This is a classification target calculated from observed signals within the snapshot. Specifically, it checks if search impressions in the last 30 days dropped by more than 20% compared to the previous 30-day window.
* **Why it is a proxy label:** The features and the label are measured on the same 90-day historical snapshot. While acceptable for a starter playground, it represents a simplified setup.
* **What the true future outcome target looks like (No-Overlap):**
  A robust capstone formulation built on the warehouse daily facts (`fact_content_daily_performance`) must use strict, non-overlapping temporal windows to avoid target leakage:
  * **Feature Window (Days -90 to Day 0):** Gather search/analytics signals (impressions, CTR, position, word count, engagement).
  * **Outcome Target Window (Days +1 to Day +30):** Observe the actual future percentage change in traffic to define whether the page went on to experience a real decline.

In [ ]:
# Compute and inspect our starter proxy label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

total_rows = len(df)
positive_cases = df['is_declining_label'].sum()
positive_pct = (positive_cases / total_rows) * 100

print(f"Target Feature: is_declining_label")
print(f"Total labeled items: {total_rows:,}")
print(f"Positive cases (Declining, trend_direction == 'down'): {positive_cases:,} ({positive_pct:.2f}%)")
print(f"Negative cases (Stable/Up/New): {total_rows - positive_cases:,} ({100 - positive_pct:.2f}%)")

Target Feature: is_declining_label
Total labeled items: 30,000
Positive cases (Declining, trend_direction == 'down'): 16,262 (54.21%)
Negative cases (Stable/Up/New): 13,738 (45.79%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

* **The Chosen Metric: Precision@K (specifically Precision@50) and Average Precision (AP)**.
* **Why this metric is defended:**
  Standard metrics like global accuracy or ROC-AUC are misleading for real-world decision-support systems. Accuracy evaluates predictions on thousands of dead or low-volume pages that editorial teams will never look at. Because editorial capacity is hard-gated (e.g., they can only review 50 pages), we must ensure that the top 50 pages recommended are highly relevant. Precision@50 measures the percentage of our top 50 recommendations that are true, high-opportunity decline/staleness candidates.
* **What 'Good' looks like:**
  Our starter baseline rules achieve a Precision@50 of **0.240** (only 12 of the top 50 recommended pages were correct). A learned Random Forest model on this starter slice elevates this to **0.740** (37 of the top 50 correct). Therefore, any acceptable ML approach must beat our heuristic baseline (0.240) and realistically aim for a Precision@50 of **>0.700** on client-holdout validation.

In [ ]:
# Evaluate naive baseline heuristics to demonstrate the difficulty of the ranking task
# Heuristic A: Rank purely by highest traffic/exposure (impressions_90d)
top_50_by_impressions = df.sort_values(by='impressions_90d', ascending=False).head(50)
p_at_50_impressions = top_50_by_impressions['is_declining_label'].mean()

# Heuristic B: Rank purely by oldest content age (content_age_days)
top_50_by_age = df.sort_values(by='content_age_days', ascending=False).head(50)
p_at_50_age = top_50_by_age['is_declining_label'].mean()

print(f"--- Baseline Heuristic Ranking Quality ---")
print(f"Precision@50 if we only recommend the highest-traffic pages: {p_at_50_impressions:.3f}")
print(f"Precision@50 if we only recommend the oldest pages: {p_at_50_age:.3f}")
print("\nThese weak scores demonstrate why flat, single-metric heuristics fail to find the right pages.")

--- Baseline Heuristic Ranking Quality ---
Precision@50 if we only recommend the highest-traffic pages: 0.420
Precision@50 if we only recommend the oldest pages: 0.760

These weak scores demonstrate why flat, single-metric heuristics fail to find the right pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

* **Definition:** The unit of analysis is **one unique pseudonymized content item (web page)** for a given client, with historical search and analytics signals aggregated over a trailing 90-day window.
* **Dataframe Grain:** The grain is uniquely defined by `content_id`. Below, we verify this grain by checking uniqueness and display the primary columns representing a single observation of our unit of analysis.

In [ ]:
# Verify the grain of our dataset
total_count = len(df)
unique_content_ids = df['content_id'].nunique()

print(f"DataFrame Row Count: {total_count:,}")
print(f"Unique Content IDs: {unique_content_ids:,}")
print(f"Is content_id unique? (Matches Row Count): {total_count == unique_content_ids}")

# Display a slice of the dataframe showing 5 content items as unit observations
representative_columns = [
    'content_id',
    'client_id',
    'impressions_90d',
    'clicks_90d',
    'ctr',
    'content_age_days',
    'days_since_last_update',
    'trend_direction'
]
df[representative_columns].head(5)

DataFrame Row Count: 30,000
Unique Content IDs: 30,000
Is content_id unique? (Matches Row Count): True


,content_id,client_id,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,187,20,down
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,445,25,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,141,20,down
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,463,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,263,14,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Fixed, hand-tuned rules (like `if impressions > 500 and age > 180 then review`) collapse on complex, multi-system data for three reasons:

1. **Non-linear and Tangled Signal Relationships:** SEO and analytics signals are deeply intertwined. For example, a low CTR is alarming if a page sits in position 2, but perfectly normal if it sits in position 9. A simple threshold rule cannot natively handle these conditional relationships without writing hundreds of messy if-statements.
2. **Heterogeneity Across Clients:** Our dataset contains 32 distinct clients. Some clients are enterprise sites with millions of monthly impressions, while others are small niche sites. A global hardcoded filter (e.g., `impressions > 1,000`) will select 100% of an enterprise client's inventory (including healthy pages) while completely missing critical, high-exposure drops on smaller clients.
3. **No Continuous Ranking Capacity:** Rules produce binary groups (Flagged vs Not Flagged). They cannot tell a team of editors *which* of the 2,000 flagged pages is the single most critical item to fix on Monday morning. Machine learning maps these complex inputs to a continuous, calibrated score, allowing for precise prioritization.

In [ ]:
# Show the extreme differences between clients to prove why static thresholds fail
client_heterogeneity = df.groupby('client_id').agg(
    total_pages=('content_id', 'count'),
    median_impressions=('impressions_90d', 'median'),
    median_ctr=('ctr', 'median')
).reset_index()

print("Client scale variability (First 10 clients):")
print(client_heterogeneity.head(10))
print("\nThis variation shows why a single global rule cannot fit all clients fairly.")

Client scale variability (First 10 clients):
           client_id  total_pages  median_impressions  median_ctr
0  client_02d20bbd7e           38                35.5        0.00
1  client_0b918943df           35                25.0        0.00
2  client_19581e27de         7008              1691.5        0.11
3  client_1a6562590e            3               342.0        0.00
4  client_25fc0e7096          476                 3.0        0.00
5  client_2c624232cd          649               228.0        0.00
6  client_349c41201b          763              3312.0        0.26
7  client_3fdba35f04         2267              1208.0        0.10
8  client_434c9b5ae5           87               318.0        0.00
9  client_4e07408562         2294              2677.5        0.17

This variation shows why a single global rule cannot fit all clients fairly.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.